<a href="https://colab.research.google.com/github/Saikadam123/ADM-Project/blob/main/CBP_Multimodal_Robustness_Model_With_Explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robust Multimodal Chest X-ray Classification with Explainability



## 1. Environment and Dependencies


In [ ]:
# Install required packages

!pip install -q transformers accelerate scikit-learn huggingface_hub safetensors --upgrade


In [ ]:
# Import libraries

import os
import gc
import json
import math
import random
import contextlib
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
)


In [ ]:
# Mount Google Drive when running in Colab

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print(
        "Not running in Google Colab -- skipping drive.mount(). "
        "Set DATASET_DIR in the configuration cell (or the CBP_DATASET_DIR environment "
        "variable) to your local dataset directory before continuing."
    )


## 2. Experiment Configuration


In [ ]:
# Define paths and hyperparameters

DATASET_DIR = os.environ.get("CBP_DATASET_DIR", "/content/drive/MyDrive/NewDataset")
IMAGE_FOLDER = os.path.join(DATASET_DIR, "images_normalized")
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
VAL_CSV = os.path.join(DATASET_DIR, "validation.csv")
TEST_CSV = os.path.join(DATASET_DIR, "test.csv")

MODEL_FOLDER = os.path.join(DATASET_DIR, "trained_model_robustness_baseline")
os.makedirs(MODEL_FOLDER, exist_ok=True)
BEST_MODEL_PATH = os.path.join(MODEL_FOLDER, "best_model.pth")

CHEXPERT_REPO_ID = "itsomk/chexpert-densenet121"
CHEXPERT_WEIGHT_FILENAMES = ("pytorch_model.safetensors", "chexpert_pytorch.safetensors")
TEXT_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

MAX_TEXT_LEN = 256
IMAGE_SIZE = 224
DENSENET_IMAGE_MEAN = [0.485, 0.456, 0.406]
DENSENET_IMAGE_STD = [0.229, 0.224, 0.225]

DENSENET_FEATURE_DIM = 1024
TEXT_DIM = 768
IMAGE_DIM = 768
CBP_PROJ_DIM = 192
CBP_CONV_FILTERS = 4
FUSION_DIM = 256
CLASSIFIER_DROPOUT = 0.3
USE_TEXT_ATTENTION_POOLING = True
USE_IMAGE_ATTENTION_POOLING = True

NUM_UNFROZEN_DENSENET_BLOCKS = 2
NUM_UNFROZEN_BERT_LAYERS = 2
PHASE1_EPOCHS = 2
PHASE2_MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

PHASE1_HEAD_LR = 1e-3
PHASE2_DENSENET_LR = 1e-5
PHASE2_BERT_LR = 8e-6
PHASE2_HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO_PHASE2 = 0.1
MAX_GRAD_NORM = 1.0

USE_OGM_GE = True
OGM_ALPHA = 3.5
OGM_MIN_COEFF = 0.02
OGM_IMAGE_BOOST = 1.35
OGM_PROTECT_IMAGE = True
OGM_GE_SIGMA = 0.05
OGM_EMA_MOMENTUM = 0.9
OGM_WARMUP_STEPS = 50

# Asymmetric modality dropout: text is removed more often to reduce shortcut reliance.
P_DROP_TEXT = 0.60
P_DROP_IMAGE = 0.05
ALLOW_BOTH_DROPPED = False
TRAIN_SUBSTITUTE = "zero"

NOISE_SIGMA_START = (0.005, 0.02)
NOISE_SIGMA_END = (0.03, 0.08)
NOISE_CURRICULUM_POWER = 1.0
APPLY_NOISE_TO_AUX = True

# Explicit loss weights; baseline FocalLoss is retained for classification terms.
W_CLEAN = 1.0
W_IMAGE_ONLY = 2.0
W_TEXT_ONLY = 0.10
W_PERTURBED = 0.0
LAMBDA_CONSISTENCY = 0.05
CONSISTENCY_BETA = 1.0
CONSISTENCY_RAMP_STEPS = 200

# Current experiment validation selection.
SELECTION_NOISE_SIGMAS = (0.05, 0.10)
SELECTION_WEIGHTS = {
    "clean_auroc": 0.55,
    "image_only_auroc": 0.15,
    "noise_auroc": 0.15,
    "unimodal_f1": 0.10,
    "image_contribution": 0.05,
}

NOISE_SWEEP_SIGMAS = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
NOISE_SWEEP_EXTENDED = [2.0, 5.0]
CALIBRATION_BATCHES = 40

PHYSICAL_BATCH_SIZE = 4
EFFECTIVE_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = max(1, EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE)
TRAIN_NUM_WORKERS = 2
EVAL_NUM_WORKERS = 0
SEED = 42
USE_AMP = True

print("Output directory:", MODEL_FOLDER)
print(f"Physical batch {PHYSICAL_BATCH_SIZE} x accumulation {GRAD_ACCUM_STEPS} = effective batch {PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS}")
print("Selection weights sum:", sum(SELECTION_WEIGHTS.values()))


In [ ]:
# Validate dataset paths

# Self-check: fail fast with a clear message if the dataset is not where DATASET_DIR expects it.
_required_paths = {
    "IMAGE_FOLDER": IMAGE_FOLDER,
    "TRAIN_CSV": TRAIN_CSV,
    "VAL_CSV": VAL_CSV,
    "TEST_CSV": TEST_CSV,
}
_missing = {name: path for name, path in _required_paths.items() if not os.path.exists(path)}
if _missing:
    raise FileNotFoundError(
        "The following required dataset paths were not found:\n"
        + "\n".join(f"  {name}: {path}" for name, path in _missing.items())
        + "\n\nIf you are not running in Google Colab, set DATASET_DIR above (or the "
          "CBP_DATASET_DIR environment variable) to the correct local dataset directory."
    )
print("All required dataset paths found under:", DATASET_DIR)


## 3. Reproducibility and Runtime Setup


In [ ]:
# Set random seed, device, and mixed precision

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and torch.cuda.is_available())
AMP_DTYPE = torch.float16 if AMP_ENABLED else torch.float32


def amp_autocast():
    if not AMP_ENABLED:
        return contextlib.nullcontext()
    return torch.amp.autocast("cuda", dtype=AMP_DTYPE)


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (TypeError, AttributeError):
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


@contextlib.contextmanager
def temporary_seed(seed: int = SEED):
    """Use deterministic evaluation noise without rewinding the training RNG stream."""
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        set_seed(seed)
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.set_rng_state(torch_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)


set_seed(SEED)
print("Device:", device)
print("AMP enabled:", AMP_ENABLED, "dtype:", AMP_DTYPE)


## 4. Dataset, Preprocessing and Data Loaders


In [ ]:
# Load dataset splits

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

required_cols = {"uid", "filename", "findings", "binary_label"}
for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing = required_cols.difference(frame.columns)
    if missing:
        raise ValueError(f"{name} CSV missing required columns: {sorted(missing)}")

print("Rows -> train:", len(train_df), "validation:", len(val_df), "test:", len(test_df))
print("Positive prevalence -> train:", float(train_df["binary_label"].mean()))


In [ ]:
# Define dataset class and data loaders

class CXRMultimodalDataset(Dataset):
    def __init__(self, df, image_folder, image_processor, tokenizer, max_text_len=256):
        self.df = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_folder, row["filename"])
        with Image.open(image_path) as img:
            image = img.convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        encoded = self.tokenizer(
            str(row["findings"]), padding="max_length", truncation=True,
            max_length=self.max_text_len, return_tensors="pt",
        )
        return {
            "pixel_values": pixel_values,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(row["binary_label"], dtype=torch.float32),
            "uid": row["uid"],
        }


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "label": torch.stack([b["label"] for b in batch]),
        "uid": [b["uid"] for b in batch],
    }


def move_batch_to_device(batch, device):
    return {
        "pixel_values": batch["pixel_values"].to(device, non_blocking=True),
        "input_ids": batch["input_ids"].to(device, non_blocking=True),
        "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
        "label": batch["label"].to(device, non_blocking=True),
    }


class DenseNetImageProcessor:
    def __init__(self, image_size=IMAGE_SIZE):
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=DENSENET_IMAGE_MEAN, std=DENSENET_IMAGE_STD),
        ])

    def __call__(self, images, return_tensors="pt"):
        if return_tensors != "pt":
            raise ValueError("DenseNetImageProcessor supports return_tensors='pt' only.")
        return {"pixel_values": self.transform(images.convert("RGB")).unsqueeze(0)}


image_processor = DenseNetImageProcessor()
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

train_dataset = CXRMultimodalDataset(train_df, IMAGE_FOLDER, image_processor, tokenizer, MAX_TEXT_LEN)
val_dataset = CXRMultimodalDataset(val_df, IMAGE_FOLDER, image_processor, tokenizer, MAX_TEXT_LEN)
test_dataset = CXRMultimodalDataset(test_df, IMAGE_FOLDER, image_processor, tokenizer, MAX_TEXT_LEN)

train_loader = DataLoader(train_dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=True,
                          num_workers=TRAIN_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                          drop_last=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=False,
                        num_workers=EVAL_NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=False,
                         num_workers=EVAL_NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)

print("DenseNet preprocessing: RGB -> 224x224 -> tensor -> checkpoint normalization")
print("Batches -> train:", len(train_loader), "validation:", len(val_loader), "test:", len(test_loader))


## 5. Multimodal Model Architecture


In [ ]:
# Define attention pooling and residual CBP

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attn(hidden_states).squeeze(-1)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1), weights.squeeze(-1)


class CompactBilinearPoolingCNN(nn.Module):
    def __init__(self, text_dim=TEXT_DIM, image_dim=IMAGE_DIM, proj_dim=CBP_PROJ_DIM,
                 conv_filters=CBP_CONV_FILTERS, fusion_dim=FUSION_DIM, dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.proj_dim = proj_dim
        self.text_proj = nn.Linear(text_dim, proj_dim)
        self.image_proj = nn.Linear(image_dim, proj_dim)
        self.proj_dropout = nn.Dropout(dropout)
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, conv_filters, kernel_size=3, padding=1),
            nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2),
        )
        pooled_dim = proj_dim // 2
        conv_out_dim = conv_filters * pooled_dim * pooled_dim
        self.fusion_proj = nn.Sequential(
            nn.Linear(conv_out_dim + proj_dim + proj_dim, fusion_dim),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
        )

    def forward(self, text_features, image_features):
        t = self.proj_dropout(self.text_proj(text_features))
        v = self.proj_dropout(self.image_proj(image_features))
        bilinear_map = torch.bmm(v.unsqueeze(2), t.unsqueeze(1)).unsqueeze(1)
        compressed = self.conv_block(bilinear_map).flatten(start_dim=1)
        return self.fusion_proj(torch.cat([compressed, t, v], dim=1))


In [ ]:
# Load and verify CheXpert DenseNet121 weights

def _download_chexpert_weights():
    last_error = None
    for filename in CHEXPERT_WEIGHT_FILENAMES:
        try:
            path = hf_hub_download(repo_id=CHEXPERT_REPO_ID, filename=filename)
            print(f"CheXpert checkpoint source: {CHEXPERT_REPO_ID}/{filename}")
            print("CheXpert checkpoint path:", path)
            return path
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not download a supported checkpoint from {CHEXPERT_REPO_ID}: {last_error}")


def build_verified_chexpert_densenet121():
    checkpoint_path = _download_chexpert_weights()
    raw_state = load_file(checkpoint_path)
    had_wrapper_prefix = any(k.startswith("densenet.") for k in raw_state)
    state_dict = {
        (k[len("densenet."):] if k.startswith("densenet.") else k): v
        for k, v in raw_state.items()
    }
    feature_state = {k: v for k, v in state_dict.items() if k.startswith("features.")}
    classifier_keys = [k for k in state_dict if k.startswith("classifier.")]
    if not feature_state:
        raise RuntimeError("No features.* tensors found in CheXpert checkpoint.")

    backbone = models.densenet121(weights=None)
    fresh = models.densenet121(weights=None)
    load_report = backbone.load_state_dict(feature_state, strict=False)
    backbone_missing = [k for k in load_report.missing_keys if k.startswith("features.")]
    unexpected = [k for k in load_report.unexpected_keys if k.startswith("features.")]
    model_feature_keys = {k for k in backbone.state_dict() if k.startswith("features.")}
    loaded_feature_keys = sorted(model_feature_keys.intersection(feature_state.keys()))

    if backbone_missing:
        raise RuntimeError(f"CheXpert DenseNet backbone failed: {len(backbone_missing)} missing features.* keys")
    if unexpected:
        raise RuntimeError(f"Unexpected CheXpert features.* keys: {unexpected[:10]}")
    if len(loaded_feature_keys) != len(model_feature_keys):
        raise RuntimeError("CheXpert DenseNet feature-key coverage is incomplete.")

    differs_from_fresh = not torch.allclose(
        backbone.state_dict()["features.conv0.weight"].detach().cpu(),
        fresh.state_dict()["features.conv0.weight"].detach().cpu(),
    )
    del fresh
    if not differs_from_fresh:
        raise RuntimeError("Loaded DenseNet unexpectedly matches fresh initialization.")

    backbone.classifier = nn.Identity()
    report = {
        "repo_id": CHEXPERT_REPO_ID,
        "checkpoint_path": checkpoint_path,
        "wrapper_prefix_removed": bool(had_wrapper_prefix),
        "checkpoint_feature_keys": len(feature_state),
        "loaded_feature_keys": len(loaded_feature_keys),
        "missing_backbone_feature_keys": len(backbone_missing),
        "unexpected_feature_keys": len(unexpected),
        "ignored_classifier_keys": len(classifier_keys),
        "differs_from_fresh_init": bool(differs_from_fresh),
    }
    print("SUCCESS: CheXpert-pretrained DenseNet121 backbone loaded.")
    print("Removed 'densenet.' prefix:", report["wrapper_prefix_removed"])
    print("Missing backbone features.* keys:", report["missing_backbone_feature_keys"])
    return backbone, report


In [ ]:
# Define the multimodal classifier

class MultimodalCBPClassifier(nn.Module):
    """Multimodal CBP classifier with robustness-compatible latent fusion hooks."""
    def __init__(self):
        super().__init__()
        self.image_encoder, self.chexpert_load_report = build_verified_chexpert_densenet121()
        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME)
        self.text_attn_pool = AttentionPooling(self.text_encoder.config.hidden_size)
        self.image_attn_pool = AttentionPooling(DENSENET_FEATURE_DIM)
        self.image_projection = nn.Sequential(
            nn.LayerNorm(DENSENET_FEATURE_DIM), nn.Linear(DENSENET_FEATURE_DIM, IMAGE_DIM)
        )
        self.cbp = CompactBilinearPoolingCNN()
        self.classifier = nn.Sequential(
            nn.Linear(FUSION_DIM, FUSION_DIM // 2), nn.ReLU(inplace=True),
            nn.Dropout(CLASSIFIER_DROPOUT), nn.Linear(FUSION_DIM // 2, 1),
        )
        # Calibrated missing-modality substitutes. Buffers are not trained parameters.
        self.register_buffer("text_missing", torch.zeros(TEXT_DIM))
        self.register_buffer("image_missing", torch.zeros(IMAGE_DIM))

    def encode_image(self, pixel_values):
        image_map = F.relu(self.image_encoder.features(pixel_values), inplace=False)
        image_tokens = image_map.flatten(2).transpose(1, 2).contiguous()
        image_pooled, _ = self.image_attn_pool(image_tokens, attention_mask=None)
        return self.image_projection(image_pooled)

    def encode(self, pixel_values, input_ids, attention_mask):
        image_features = self.encode_image(pixel_values)
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_features, _ = self.text_attn_pool(text_out.last_hidden_state, attention_mask=attention_mask)
        if text_features.shape[-1] != TEXT_DIM or image_features.shape[-1] != IMAGE_DIM:
            raise RuntimeError("Unexpected modality embedding dimension.")
        return text_features, image_features

    def fuse_and_classify(self, text_features, image_features, modality_mask=None,
                          noise_sigma=0.0, substitute="zero", return_extras=False):
        if substitute not in ("zero", "calibrated"):
            raise ValueError("substitute must be 'zero' or 'calibrated'")
        b = text_features.shape[0]
        if modality_mask is None:
            modality_mask = torch.ones(b, 2, device=text_features.device, dtype=text_features.dtype)
        modality_mask = modality_mask.to(text_features.dtype)

        if substitute == "zero":
            t_sub, v_sub = torch.zeros_like(text_features), torch.zeros_like(image_features)
        else:
            t_sub = self.text_missing.to(text_features.dtype).unsqueeze(0).expand_as(text_features)
            v_sub = self.image_missing.to(image_features.dtype).unsqueeze(0).expand_as(image_features)

        mt, mv = modality_mask[:, 0:1], modality_mask[:, 1:2]
        t = mt * text_features + (1.0 - mt) * t_sub
        v = mv * image_features + (1.0 - mv) * v_sub
        t = apply_relative_gaussian_noise(t, noise_sigma)
        v = apply_relative_gaussian_noise(v, noise_sigma)
        fused = self.cbp(t, v)
        logits = self.classifier(fused).squeeze(-1)
        if return_extras:
            return logits, {"fused": fused, "text": t, "image": v}
        return logits

    def forward(self, pixel_values, input_ids, attention_mask, modality_mask=None,
                noise_sigma=0.0, substitute="zero"):
        text_features, image_features = self.encode(pixel_values, input_ids, attention_mask)
        return self.fuse_and_classify(text_features, image_features, modality_mask,
                                      noise_sigma, substitute)


def constant_mask(batch_size, text_present, image_present):
    return torch.tensor([[text_present, image_present]], device=device, dtype=torch.float32).expand(batch_size, 2)


## 6. Robustness Training Mechanisms


In [ ]:
# Define modality dropout, latent noise, and masking utilities

def sample_asymmetric_modality_mask(batch_size, p_drop_text=P_DROP_TEXT, p_drop_image=P_DROP_IMAGE,
                                    device=device, allow_both_dropped=ALLOW_BOTH_DROPPED):
    if p_drop_text < 0 or p_drop_image < 0:
        raise ValueError("Drop probabilities must be non-negative")
    if not allow_both_dropped:
        if p_drop_text + p_drop_image > 1.0:
            raise ValueError("p_drop_text + p_drop_image must be <= 1")
        r = torch.rand(batch_size, device=device)
        text_missing = r < p_drop_text
        image_missing = (r >= p_drop_text) & (r < p_drop_text + p_drop_image)
        return torch.stack([(~text_missing).float(), (~image_missing).float()], dim=1)
    drop_text = torch.rand(batch_size, device=device) < p_drop_text
    drop_image = torch.rand(batch_size, device=device) < p_drop_image
    return torch.stack([(~drop_text).float(), (~drop_image).float()], dim=1)


def apply_relative_gaussian_noise(x, sigma):
    if sigma is None or sigma <= 0:
        return x
    scale = x.detach().std(dim=-1, keepdim=True).clamp_min(1e-6)
    return x + sigma * scale * torch.randn_like(x)


class NoiseCurriculum:
    def __init__(self, total_steps, start_range=NOISE_SIGMA_START, end_range=NOISE_SIGMA_END,
                 power=NOISE_CURRICULUM_POWER):
        self.total_steps = max(int(total_steps), 1)
        self.start_range, self.end_range, self.power = start_range, end_range, float(power)

    def range_at(self, step):
        p = min(max(step / self.total_steps, 0.0), 1.0) ** self.power
        lo = self.start_range[0] + p * (self.end_range[0] - self.start_range[0])
        hi = self.start_range[1] + p * (self.end_range[1] - self.start_range[1])
        return float(lo), float(hi)

    def sample(self, step):
        lo, hi = self.range_at(step)
        return float(np.random.uniform(lo, hi))


In [ ]:
# Define OGM-GE gradient modulation

@dataclass
class ModulationState:
    coeff: float = 1.0
    ge_sigma: float = 0.0
    enabled: bool = False


class _GradientModulationFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, state):
        ctx.state = state
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        state = ctx.state
        if not state.enabled:
            return grad_output, None
        grad = grad_output * state.coeff
        if state.ge_sigma > 0:
            std = grad.detach().float().std()
            if torch.isfinite(std) and float(std) > 0:
                grad = grad + torch.randn_like(grad) * std.to(grad.dtype) * state.ge_sigma
        return grad, None


def modulate_gradient(x, state):
    return _GradientModulationFn.apply(x, state)


class OGMGEController:
    def __init__(self, alpha=OGM_ALPHA, min_coeff=OGM_MIN_COEFF, image_boost=OGM_IMAGE_BOOST,
                 ge_sigma=OGM_GE_SIGMA, ema_momentum=OGM_EMA_MOMENTUM,
                 warmup_steps=OGM_WARMUP_STEPS, protect_image=OGM_PROTECT_IMAGE,
                 enabled=USE_OGM_GE):
        self.alpha, self.min_coeff, self.image_boost = float(alpha), float(min_coeff), float(image_boost)
        self.ge_sigma, self.ema_momentum = float(ge_sigma), float(ema_momentum)
        self.warmup_steps, self.protect_image, self.enabled = int(warmup_steps), bool(protect_image), bool(enabled)
        self.text_state, self.image_state = ModulationState(), ModulationState()
        self.ratio_ema = None
        self.history = []

    @staticmethod
    def _confidence(logits, labels):
        p = torch.sigmoid(logits.detach().float())
        y = labels.detach().float()
        return float((y * p + (1 - y) * (1 - p)).mean())

    def step(self, text_logits, image_logits, labels, global_step):
        s_text = self._confidence(text_logits, labels)
        s_image = self._confidence(image_logits, labels)
        ratio = s_text / max(s_image, 1e-6)
        self.ratio_ema = ratio if self.ratio_ema is None else self.ema_momentum * self.ratio_ema + (1-self.ema_momentum) * ratio
        rho = float(self.ratio_ema)
        active = self.enabled and global_step >= self.warmup_steps
        k_text, k_image = 1.0, 1.0
        if active:
            if rho > 1.0:
                k_text = 1.0 - math.tanh(self.alpha * (rho - 1.0))
            else:
                k_image = 1.0 - math.tanh(self.alpha * (1.0 / max(rho, 1e-6) - 1.0))
            k_text = float(np.clip(k_text, self.min_coeff, 1.0))
            k_image = float(np.clip(k_image, self.min_coeff, 1.0))
            if self.protect_image:
                k_image = 1.0
            k_image *= self.image_boost
        for state, coeff in ((self.text_state, k_text), (self.image_state, k_image)):
            state.coeff, state.ge_sigma, state.enabled = coeff, (self.ge_sigma if active else 0.0), active
        rec = {"step": float(global_step), "score_text": s_text, "score_image": s_image,
               "ratio": ratio, "ratio_ema": rho, "k_text": k_text, "k_image": k_image,
               "active": float(active)}
        self.history.append(rec)
        return rec

    def history_frame(self):
        return pd.DataFrame(self.history)


# OGM-GE self-tests: forward identity, backward scaling, and activation under text dominance.
_state = ModulationState(coeff=0.25, enabled=True)
_x = torch.randn(8, 16, requires_grad=True)
_y = modulate_gradient(_x, _state)
assert torch.equal(_x, _y)
_y.sum().backward()
assert torch.allclose(_x.grad, torch.full_like(_x.grad, 0.25))

_ctrl = OGMGEController(warmup_steps=0)
_labels = torch.randint(0, 2, (64,)).float()
_text_logits = torch.where(_labels > 0, torch.full((64,), 4.0), torch.full((64,), -4.0))
_image_logits = torch.zeros(64)
_rec = _ctrl.step(_text_logits, _image_logits, _labels, global_step=10)
assert _rec["k_text"] < 1.0 and _rec["k_image"] >= 1.0
print("OGM-GE self-tests passed.")


## 7. Training Objective


In [ ]:
# Define focal loss and robustness objective

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal = (1 - p_t).clamp(min=1e-6) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * focal * bce
        else:
            loss = focal * bce
        return loss.mean() if self.reduction == "mean" else loss.sum() if self.reduction == "sum" else loss


pos_frac = float(train_df["binary_label"].mean())
FOCAL_ALPHA = float(1.0 - pos_frac)
FOCAL_GAMMA = 2.0
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)


def consistency_weight(global_step):
    if CONSISTENCY_RAMP_STEPS <= 0:
        return LAMBDA_CONSISTENCY
    return LAMBDA_CONSISTENCY * min(1.0, global_step / float(CONSISTENCY_RAMP_STEPS))


def compute_total_loss(logits, labels, lam_cons):
    y = labels.float()
    loss_clean = criterion(logits["clean"].float(), y)
    loss_img = criterion(logits["image_only"].float(), y)
    loss_txt = criterion(logits["text_only"].float(), y)
    loss_cons = F.smooth_l1_loss(logits["perturbed"].float(), logits["clean"].float().detach(), beta=CONSISTENCY_BETA)
    loss_pert = criterion(logits["perturbed"].float(), y) if W_PERTURBED > 0 else torch.zeros((), device=y.device)
    total = (W_CLEAN * loss_clean + W_IMAGE_ONLY * loss_img + W_TEXT_ONLY * loss_txt +
             W_PERTURBED * loss_pert + lam_cons * loss_cons)
    parts = {"loss_clean": float(loss_clean.detach()), "loss_img": float(loss_img.detach()),
             "loss_txt": float(loss_txt.detach()), "loss_pert": float(loss_pert.detach()),
             "loss_cons": float(loss_cons.detach()), "lambda": float(lam_cons)}
    return total, parts

print(f"FocalLoss alpha={FOCAL_ALPHA:.4f}, gamma={FOCAL_GAMMA}")
print("Robustness loss weights:", W_CLEAN, W_IMAGE_ONLY, W_TEXT_ONLY, W_PERTURBED, LAMBDA_CONSISTENCY)


## 8. Model Initialization and Architecture Verification


In [ ]:
# Instantiate and verify the model

model = MultimodalCBPClassifier().to(device)
report = model.chexpert_load_report
assert report["missing_backbone_feature_keys"] == 0
assert report["unexpected_feature_keys"] == 0
assert report["differs_from_fresh_init"]
assert isinstance(model.image_encoder.classifier, nn.Identity)
assert model.image_projection[1].in_features == 1024 and model.image_projection[1].out_features == 768
assert model.text_encoder.config.hidden_size == 768
assert model.cbp.proj_dim == 192
assert model.classifier[-1].out_features == 1

print("Baseline architecture assertions passed:")
print("  DenseNet feature dim =", DENSENET_FEATURE_DIM)
print("  image projection = 1024 ->", IMAGE_DIM)
print("  BioClinicalBERT hidden dim =", TEXT_DIM)
print("  CBP projection dim =", CBP_PROJ_DIM)
print("  fusion dim =", FUSION_DIM)
print("  classifier output = 1")


## 9. Fine-tuning Setup and Optimizer Groups


In [ ]:
# Define freezing, unfreezing, and optimizer utilities

def freeze_all_backbone_params(model):
    for p in model.image_encoder.parameters(): p.requires_grad = False
    for p in model.text_encoder.parameters(): p.requires_grad = False


def unfreeze_top_layers(model, num_densenet_blocks=NUM_UNFROZEN_DENSENET_BLOCKS,
                        num_bert_layers=NUM_UNFROZEN_BERT_LAYERS):
    f = model.image_encoder.features
    blocks = [(f.denseblock1, f.transition1), (f.denseblock2, f.transition2),
              (f.denseblock3, f.transition3), (f.denseblock4, f.norm5)]
    for pair in blocks[-min(num_densenet_blocks, len(blocks)):]:
        for module in pair:
            for p in module.parameters(): p.requires_grad = True
    bert_layers = model.text_encoder.encoder.layer
    for layer in bert_layers[-min(num_bert_layers, len(bert_layers)):]:
        for p in layer.parameters(): p.requires_grad = True
    if getattr(model.text_encoder, "pooler", None) is not None:
        for p in model.text_encoder.pooler.parameters(): p.requires_grad = True


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())


def build_phase2_param_groups(model):
    image_params, text_params, head_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if name.startswith("image_encoder"):
            image_params.append(p)
        elif name.startswith("text_encoder"):
            text_params.append(p)
        else:
            head_params.append(p)
    return [
        {"params": image_params, "lr": PHASE2_DENSENET_LR, "weight_decay": WEIGHT_DECAY},
        {"params": text_params, "lr": PHASE2_BERT_LR, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": PHASE2_HEAD_LR, "weight_decay": WEIGHT_DECAY},
    ]


freeze_all_backbone_params(model)
scaler = make_grad_scaler()

# One complete forward/backward safety batch before full training.
batch = next(iter(train_loader))
moved = move_batch_to_device(batch, device)
model.train(); model.zero_grad(set_to_none=True)
with amp_autocast():
    text_features, image_features = model.encode(moved["pixel_values"], moved["input_ids"], moved["attention_mask"])
    bs = moved["label"].shape[0]
    logits_dict = {
        "clean": model.fuse_and_classify(text_features, image_features),
        "image_only": model.fuse_and_classify(text_features, image_features, constant_mask(bs, 0, 1)),
        "text_only": model.fuse_and_classify(text_features, image_features, constant_mask(bs, 1, 0)),
        "perturbed": model.fuse_and_classify(text_features, image_features, sample_asymmetric_modality_mask(bs, device=device)),
    }
    safety_loss, _ = compute_total_loss(logits_dict, moved["label"], 0.0)
assert torch.isfinite(logits_dict["clean"]).all() and torch.isfinite(safety_loss)
safety_loss.backward()
for name, module in [("image_projection", model.image_projection), ("CBP", model.cbp), ("classifier", model.classifier)]:
    grad = next(module.parameters()).grad
    assert grad is not None and torch.isfinite(grad).all(), f"Bad gradient in {name}"
print("Pre-training forward/backward checks passed.")
print("Text embedding:", tuple(text_features.shape), "Image embedding:", tuple(image_features.shape), "Logits:", tuple(logits_dict["clean"].shape))
model.zero_grad(set_to_none=True)
del batch, moved, text_features, image_features, logits_dict, safety_loss
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 10. Training and Evaluation Functions


In [ ]:
# Define evaluation metrics and validation helpers

MASKS = {"multimodal": (1.0, 1.0), "text_only": (1.0, 0.0), "image_only": (0.0, 1.0)}


def compute_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, zero_division=0),
        "Recall": recall_score(labels, preds, zero_division=0),
        "F1": f1_score(labels, preds, zero_division=0),
        "MCC": matthews_corrcoef(labels, preds) if len(np.unique(preds)) > 1 else 0.0,
        "AUROC": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan"),
    }


def tune_threshold(labels, probs):
    candidates = np.linspace(0.05, 0.95, 181)
    scores = [f1_score(labels, (probs >= t).astype(int), zero_division=0) for t in candidates]
    i = int(np.argmax(scores))
    return float(candidates[i]), float(scores[i])


@torch.no_grad()
def evaluate_conditions(model, loader, noise_sigmas=SELECTION_NOISE_SIGMAS, threshold=0.5,
                        substitute="zero", seed=SEED, collect_uids=False):
    model.eval()
    labels_all, uids_all = [], []
    probs = {k: [] for k in MASKS}
    for sigma in noise_sigmas: probs[f"noise_{sigma:g}"] = []

    with temporary_seed(seed):
        for batch in loader:
            moved = move_batch_to_device(batch, device)
            labels_all.append(moved["label"].cpu().numpy())
            if collect_uids: uids_all.extend(batch["uid"])
            with amp_autocast():
                t, v = model.encode(moved["pixel_values"], moved["input_ids"], moved["attention_mask"])
                b = t.shape[0]
                for cond, (mt, mv) in MASKS.items():
                    lg = model.fuse_and_classify(t, v, constant_mask(b, mt, mv), 0.0, substitute)
                    probs[cond].append(torch.sigmoid(lg).float().cpu().numpy())
                for sigma in noise_sigmas:
                    lg = model.fuse_and_classify(t, v, constant_mask(b, 1, 1), float(sigma), substitute)
                    probs[f"noise_{sigma:g}"].append(torch.sigmoid(lg).float().cpu().numpy())

    labels = np.concatenate(labels_all).ravel()
    probs = {k: np.concatenate(v).ravel() for k, v in probs.items()}
    metrics = {k: compute_metrics(labels, p, threshold) for k, p in probs.items()}
    return metrics, labels, probs, uids_all


def composite_selection_score(metrics):
    clean = metrics["multimodal"]["AUROC"]
    image = metrics["image_only"]["AUROC"]
    text = metrics["text_only"]["AUROC"]
    noise = float(np.mean([metrics[f"noise_{s:g}"]["AUROC"] for s in SELECTION_NOISE_SIGMAS]))
    unimodal_f1 = 0.5 * (metrics["image_only"]["F1"] + metrics["text_only"]["F1"])
    # Map the ablation drop C-T into [0,1] without forcing any target attribution ratio.
    contribution_component = float(np.clip(0.5 + 5.0 * (clean - text), 0.0, 1.0))
    score = (SELECTION_WEIGHTS["clean_auroc"] * clean +
             SELECTION_WEIGHTS["image_only_auroc"] * image +
             SELECTION_WEIGHTS["noise_auroc"] * noise +
             SELECTION_WEIGHTS["unimodal_f1"] * unimodal_f1 +
             SELECTION_WEIGHTS["image_contribution"] * contribution_component)
    details = {"clean_auroc": clean, "text_only_auroc": text, "image_only_auroc": image,
               "noise_auroc": noise, "unimodal_f1": unimodal_f1,
               "image_ablation_drop": clean - text, "selection_score": score}
    return float(score), details


In [ ]:
# Define robust training loop

def train_one_epoch_robust(model, loader, optimizer, scaler, ogm, curriculum,
                           scheduler=None, accum_steps=GRAD_ACCUM_STEPS,
                           global_step=0, desc="Training"):
    model.train(); optimizer.zero_grad(set_to_none=True)
    totals = {k: 0.0 for k in ("total", "loss_clean", "loss_img", "loss_txt", "loss_pert", "loss_cons")}
    sigmas, k_texts, k_images, ratios = [], [], [], []

    for step, batch in enumerate(tqdm(loader, desc=desc, leave=False)):
        moved = move_batch_to_device(batch, device)
        labels, bs = moved["label"], moved["label"].shape[0]
        sigma = curriculum.sample(global_step)
        aux_sigma = sigma if APPLY_NOISE_TO_AUX else 0.0
        perturbed_mask = sample_asymmetric_modality_mask(bs, device=device)

        with amp_autocast():
            text_raw, image_raw = model.encode(moved["pixel_values"], moved["input_ids"], moved["attention_mask"])
            # Attach modulation immediately after the modality encoders/pooling/projection.
            text_features = modulate_gradient(text_raw, ogm.text_state)
            image_features = modulate_gradient(image_raw, ogm.image_state)
            logits = {
                "clean": model.fuse_and_classify(text_features, image_features, None, 0.0, TRAIN_SUBSTITUTE),
                "image_only": model.fuse_and_classify(text_features, image_features, constant_mask(bs, 0, 1), aux_sigma, TRAIN_SUBSTITUTE),
                "text_only": model.fuse_and_classify(text_features, image_features, constant_mask(bs, 1, 0), aux_sigma, TRAIN_SUBSTITUTE),
                "perturbed": model.fuse_and_classify(text_features, image_features, perturbed_mask, sigma, TRAIN_SUBSTITUTE),
            }
            loss, parts = compute_total_loss(logits, labels, consistency_weight(global_step))
            scaled_loss = loss / accum_steps

        # OGM-GE is updated after forward but before backward, so this batch's gradient is modulated.
        rec = ogm.step(logits["text_only"], logits["image_only"], labels, global_step)
        scaler.scale(scaled_loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            if scheduler is not None: scheduler.step()
            global_step += 1

        totals["total"] += float(loss.detach())
        for k in ("loss_clean", "loss_img", "loss_txt", "loss_pert", "loss_cons"): totals[k] += parts[k]
        sigmas.append(sigma); k_texts.append(rec["k_text"]); k_images.append(rec["k_image"]); ratios.append(rec["ratio_ema"])

    n = max(len(loader), 1)
    summary = {k: v/n for k, v in totals.items()}
    summary.update({"sigma": float(np.mean(sigmas)), "k_text": float(np.mean(k_texts)),
                    "k_image": float(np.mean(k_images)), "ogm_ratio": float(np.mean(ratios))})
    return summary, global_step


## 11. Phase 1 — Frozen Backbone Training


In [ ]:
# Train Phase 1

# Number of optimizer steps expected across both phases drives the latent-noise curriculum.
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_planned_steps = steps_per_epoch * (PHASE1_EPOCHS + PHASE2_MAX_EPOCHS)
curriculum = NoiseCurriculum(total_planned_steps)
ogm = OGMGEController()

global_step = 0
best_score = -np.inf
best_epoch = None
best_phase = None
best_details = None
training_history = []
validation_history = []

phase1_params = [p for p in model.parameters() if p.requires_grad]
phase1_optimizer = torch.optim.AdamW(phase1_params, lr=PHASE1_HEAD_LR, weight_decay=WEIGHT_DECAY)

for epoch in range(1, PHASE1_EPOCHS + 1):
    train_summary, global_step = train_one_epoch_robust(
        model, train_loader, phase1_optimizer, scaler, ogm, curriculum,
        global_step=global_step, desc=f"Phase 1 epoch {epoch}"
    )
    val_metrics, _, _, _ = evaluate_conditions(model, val_loader)
    score, details = composite_selection_score(val_metrics)
    training_history.append({"phase": 1, "epoch": epoch, **train_summary})
    validation_history.append({"phase": 1, "epoch": epoch, **details})
    print(f"Phase 1 epoch {epoch}: clean={details['clean_auroc']:.4f} image={details['image_only_auroc']:.4f} "
          f"text={details['text_only_auroc']:.4f} noise={details['noise_auroc']:.4f} score={score:.4f}")
    if score > best_score:
        best_score, best_epoch, best_phase, best_details = score, epoch, 1, details
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("  -> saved new validation-selected checkpoint")


## 12. Phase 2 — Partial Backbone Fine-tuning with discriminative learning rates


In [ ]:
# Train Phase 2

unfreeze_top_layers(model)
trainable, total = count_trainable_params(model)
print(f"Phase 2 trainable parameters: {trainable:,}/{total:,} ({100*trainable/total:.2f}%)")

phase2_optimizer = torch.optim.AdamW(build_phase2_param_groups(model))
phase2_total_steps = steps_per_epoch * PHASE2_MAX_EPOCHS
phase2_warmup_steps = max(1, int(WARMUP_RATIO_PHASE2 * phase2_total_steps))
phase2_scheduler = get_cosine_schedule_with_warmup(
    phase2_optimizer, num_warmup_steps=phase2_warmup_steps, num_training_steps=phase2_total_steps
)

epochs_without_improvement = 0
for epoch in range(1, PHASE2_MAX_EPOCHS + 1):
    train_summary, global_step = train_one_epoch_robust(
        model, train_loader, phase2_optimizer, scaler, ogm, curriculum,
        scheduler=phase2_scheduler, global_step=global_step, desc=f"Phase 2 epoch {epoch}"
    )
    val_metrics, _, _, _ = evaluate_conditions(model, val_loader)
    score, details = composite_selection_score(val_metrics)
    training_history.append({"phase": 2, "epoch": epoch, **train_summary})
    validation_history.append({"phase": 2, "epoch": epoch, **details})
    print(f"Phase 2 epoch {epoch}: clean={details['clean_auroc']:.4f} image={details['image_only_auroc']:.4f} "
          f"text={details['text_only_auroc']:.4f} noise={details['noise_auroc']:.4f} score={score:.4f}")
    if score > best_score:
        best_score, best_epoch, best_phase, best_details = score, epoch, 2, details
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("  -> saved new validation-selected checkpoint")
    else:
        epochs_without_improvement += 1
        print("  -> no validation-score improvement for", epochs_without_improvement, "epoch(s)")
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

pd.DataFrame(training_history).to_csv(os.path.join(MODEL_FOLDER, "training_history.csv"), index=False)
pd.DataFrame(validation_history).to_csv(os.path.join(MODEL_FOLDER, "validation_history.csv"), index=False)
ogm.history_frame().to_csv(os.path.join(MODEL_FOLDER, "ogm_history.csv"), index=False)
print("Best validation checkpoint:", BEST_MODEL_PATH)
print("Best phase/epoch:", best_phase, best_epoch, "score:", best_score)


## 13. Best Checkpoint Loading and Validation Threshold Selection


In [ ]:
# Load best checkpoint and tune validation threshold

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
val_metrics, val_labels, val_probs, _ = evaluate_conditions(model, val_loader, noise_sigmas=())
best_threshold, best_val_f1 = tune_threshold(val_labels, val_probs["multimodal"])
print(f"Validation-selected threshold: {best_threshold:.3f} (multimodal F1={best_val_f1:.4f})")

validation_rows = []
for cond in MASKS:
    m = compute_metrics(val_labels, val_probs[cond], best_threshold)
    validation_rows.append({"Condition": cond, "Threshold": best_threshold, **m})
validation_results_df = pd.DataFrame(validation_rows)
display(validation_results_df)


## 14. Missing-Modality Substitute Calibration


In [ ]:
# Calibrate optional missing-modality substitutes

@torch.no_grad()
def calibrate_missing_substitutes(model, loader, max_batches=CALIBRATION_BATCHES):
    model.eval(); text_sum = None; image_sum = None; n = 0
    for i, batch in enumerate(loader):
        if i >= max_batches: break
        moved = move_batch_to_device(batch, device)
        with amp_autocast():
            t, v = model.encode(moved["pixel_values"], moved["input_ids"], moved["attention_mask"])
        text_sum = t.float().sum(0) if text_sum is None else text_sum + t.float().sum(0)
        image_sum = v.float().sum(0) if image_sum is None else image_sum + v.float().sum(0)
        n += t.shape[0]
    if n == 0: raise RuntimeError("No training samples available for missing-modality calibration")
    model.text_missing.copy_((text_sum / n).to(model.text_missing.device, model.text_missing.dtype))
    model.image_missing.copy_((image_sum / n).to(model.image_missing.device, model.image_missing.dtype))
    print("Calibrated missing-modality substitutes from", n, "training samples only.")

calibrate_missing_substitutes(model, train_loader)


## 15. Held-out Test Evaluation


In [ ]:
# Evaluate clean and missing-modality test conditions

test_metrics_zero, test_labels, test_probs_zero, test_uids = evaluate_conditions(
    model, test_loader, noise_sigmas=(), threshold=best_threshold, substitute="zero", collect_uids=True
)
test_metrics_cal, _, test_probs_cal, _ = evaluate_conditions(
    model, test_loader, noise_sigmas=(), threshold=best_threshold, substitute="calibrated", collect_uids=False
)

missing_rows = []
for substitute_name, metrics in [("zero", test_metrics_zero), ("calibrated", test_metrics_cal)]:
    for cond in MASKS:
        missing_rows.append({"Substitution": substitute_name, "Condition": cond, **metrics[cond]})
missing_modality_results_df = pd.DataFrame(missing_rows)
display(missing_modality_results_df)
missing_modality_results_df.to_csv(os.path.join(MODEL_FOLDER, "missing_modality_results.csv"), index=False)

# Primary clean test report uses the training-time zero-substitution policy.
clean_test = test_metrics_zero["multimodal"]
cm = confusion_matrix(test_labels, (test_probs_zero["multimodal"] >= best_threshold).astype(int))
print("Clean held-out test metrics:")
print(pd.DataFrame([{"Threshold": best_threshold, **clean_test}]).to_string(index=False))
print("Confusion matrix:\n", cm)


## 16. Modality Contribution Analysis


In [ ]:
# Compute modality contribution statistics

C = float(test_metrics_zero["multimodal"]["AUROC"])
T = float(test_metrics_zero["text_only"]["AUROC"])
I = float(test_metrics_zero["image_only"]["AUROC"])
image_drop = C - T
text_drop = C - I
den = image_drop + text_drop
image_attribution = 100.0 * image_drop / den if abs(den) > 1e-12 else float("nan")
text_attribution = 100.0 * text_drop / den if abs(den) > 1e-12 else float("nan")

attribution_results_df = pd.DataFrame([{
    "Clean multimodal AUROC": C,
    "Text-only AUROC": T,
    "Image-only AUROC": I,
    "Image ablation drop (C-T)": image_drop,
    "Text ablation drop (C-I)": text_drop,
    "Image attribution %": image_attribution,
    "Text attribution %": text_attribution,
}])
display(attribution_results_df)
attribution_results_df.to_csv(os.path.join(MODEL_FOLDER, "attribution_results.csv"), index=False)


## 17. Latent-Noise Robustness Evaluation


In [ ]:
# Evaluate latent-noise robustness

@torch.no_grad()
def cache_latents(model, loader):
    model.eval(); T, V, Y, U = [], [], [], []
    for batch in tqdm(loader, desc="Caching test latents", leave=False):
        moved = move_batch_to_device(batch, device)
        with amp_autocast(): t, v = model.encode(moved["pixel_values"], moved["input_ids"], moved["attention_mask"])
        T.append(t.float().cpu()); V.append(v.float().cpu()); Y.append(moved["label"].float().cpu()); U.extend(batch["uid"])
    return torch.cat(T), torch.cat(V), torch.cat(Y).numpy().ravel(), U


@torch.no_grad()
def probs_from_latents(model, text_latents, image_latents, condition="multimodal", noise_sigma=0.0,
                       substitute="zero", seed=SEED, chunk=64):
    mt, mv = MASKS[condition]; out = []
    with temporary_seed(seed):
        for i in range(0, text_latents.shape[0], chunk):
            t = text_latents[i:i+chunk].to(device); v = image_latents[i:i+chunk].to(device)
            with amp_autocast():
                lg = model.fuse_and_classify(t, v, constant_mask(t.shape[0], mt, mv), noise_sigma, substitute)
            out.append(torch.sigmoid(lg).float().cpu().numpy())
    return np.concatenate(out).ravel()


test_T, test_V, cached_test_y, cached_test_uids = cache_latents(model, test_loader)
base_probs = probs_from_latents(model, test_T, test_V, noise_sigma=0.0)
noise_rows = []
for sigma in sorted(set(NOISE_SWEEP_SIGMAS) | set(NOISE_SWEEP_EXTENDED)):
    p = probs_from_latents(model, test_T, test_V, noise_sigma=float(sigma))
    m = compute_metrics(cached_test_y, p, best_threshold)
    shift = np.abs(p - base_probs)
    noise_rows.append({"sigma": sigma, **m, "Mean Abs Prob Shift": float(shift.mean()),
                       "Max Abs Prob Shift": float(shift.max())})
noise_sweep_results_df = pd.DataFrame(noise_rows)
display(noise_sweep_results_df)
noise_sweep_results_df.to_csv(os.path.join(MODEL_FOLDER, "noise_sweep_results.csv"), index=False)


## 18. OGM-GE Training Diagnostics


In [ ]:
# Summarize OGM-GE diagnostics

ogm_history_df = ogm.history_frame()
if len(ogm_history_df):
    active = ogm_history_df[ogm_history_df["active"] > 0]
    ogm_summary = {
        "OGM-GE active percentage": 100.0 * float(ogm_history_df["active"].mean()),
        "Mean text confidence": float(ogm_history_df["score_text"].mean()),
        "Mean image confidence": float(ogm_history_df["score_image"].mean()),
        "Final discrepancy ratio EMA": float(ogm_history_df["ratio_ema"].iloc[-1]),
        "Mean active text coefficient": float(active["k_text"].mean()) if len(active) else 1.0,
        "Minimum text coefficient": float(ogm_history_df["k_text"].min()),
        "Mean image coefficient": float(ogm_history_df["k_image"].mean()),
    }
else:
    ogm_summary = {"OGM-GE active percentage": 0.0, "Mean active text coefficient": 1.0,
                   "Minimum text coefficient": 1.0, "Mean image coefficient": 1.0}
print(pd.DataFrame([ogm_summary]).to_string(index=False))


## 19. Representation Diagnostics


In [ ]:
# Summarize latent representation norms

@torch.no_grad()
def norm_statistics(model, text_latents, image_latents, substitute="zero", chunk=64):
    rows = []
    for cond, (mt, mv) in MASKS.items():
        fused_norms, text_norms, image_norms = [], [], []
        for i in range(0, text_latents.shape[0], chunk):
            t = text_latents[i:i+chunk].to(device); v = image_latents[i:i+chunk].to(device)
            with amp_autocast():
                _, extras = model.fuse_and_classify(t, v, constant_mask(t.shape[0], mt, mv),
                                                    0.0, substitute, return_extras=True)
            fused_norms.append(extras["fused"].float().norm(dim=-1).cpu())
            text_norms.append(extras["text"].float().norm(dim=-1).cpu())
            image_norms.append(extras["image"].float().norm(dim=-1).cpu())
        rows.append({"Condition": cond,
                     "Mean text latent norm": float(torch.cat(text_norms).mean()),
                     "Mean image latent norm": float(torch.cat(image_norms).mean()),
                     "Mean fused norm": float(torch.cat(fused_norms).mean())})
    df = pd.DataFrame(rows)
    mm = float(df.loc[df.Condition == "multimodal", "Mean fused norm"].iloc[0])
    df["Fused norm retention vs multimodal"] = df["Mean fused norm"] / max(mm, 1e-12)
    return df

norm_diagnostics_df = norm_statistics(model, test_T, test_V, substitute="zero")
display(norm_diagnostics_df)
norm_diagnostics_df.to_csv(os.path.join(MODEL_FOLDER, "norm_diagnostics.csv"), index=False)


## 20. Save Final Predictions and Results


In [ ]:
# Save final test predictions and consolidated results

test_predictions_df = pd.DataFrame({
    "uid": cached_test_uids,
    "true_label": cached_test_y.astype(int),
    "prob_multimodal": probs_from_latents(model, test_T, test_V, "multimodal", 0.0),
    "prob_text_only": probs_from_latents(model, test_T, test_V, "text_only", 0.0),
    "prob_image_only": probs_from_latents(model, test_T, test_V, "image_only", 0.0),
})
test_predictions_df["pred_multimodal"] = (test_predictions_df["prob_multimodal"] >= best_threshold).astype(int)
test_predictions_df.to_csv(os.path.join(MODEL_FOLDER, "test_predictions.csv"), index=False)

noise_lookup = noise_sweep_results_df.set_index("sigma")
summary_rows = [
    ("Clean multimodal AUROC", C),
    ("Clean multimodal F1", clean_test["F1"]),
    ("Text-only AUROC", T),
    ("Image-only AUROC", I),
    ("Image contribution (C-T)", image_drop),
    ("Image attribution %", image_attribution),
]
for sigma in NOISE_SWEEP_SIGMAS[1:]:
    summary_rows.append((f"AUROC @ sigma={sigma:g}", float(noise_lookup.loc[sigma, "AUROC"])))
summary_rows.extend([
    ("Best validation phase", best_phase),
    ("Best validation epoch", best_epoch),
    ("Best validation composite score", best_score),
    ("OGM-GE active percentage", ogm_summary.get("OGM-GE active percentage", 0.0)),
    ("Mean active text coefficient", ogm_summary.get("Mean active text coefficient", 1.0)),
    ("Minimum text coefficient", ogm_summary.get("Minimum text coefficient", 1.0)),
    ("Mean image coefficient", ogm_summary.get("Mean image coefficient", 1.0)),
])
final_summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Result"])
display(final_summary_df)

summary = {
    "checkpoint": BEST_MODEL_PATH,
    "best_validation_phase": int(best_phase),
    "best_validation_epoch": int(best_epoch),
    "best_validation_composite_score": float(best_score),
    "validation_selected_threshold": float(best_threshold),
    "architecture": {"densenet_feature_dim": 1024, "image_dim": 768, "text_dim": 768,
                     "cbp_proj_dim": 192, "fusion_dim": 256},
    "training": {"phase1_head_lr": PHASE1_HEAD_LR, "phase2_densenet_lr": PHASE2_DENSENET_LR,
                 "phase2_bert_lr": PHASE2_BERT_LR, "phase2_head_lr": PHASE2_HEAD_LR,
                 "physical_batch_size": PHYSICAL_BATCH_SIZE, "gradient_accumulation": GRAD_ACCUM_STEPS},
    "results": {"clean_auroc": C, "clean_f1": float(clean_test["F1"]),
                "text_only_auroc": T, "image_only_auroc": I,
                "image_ablation_drop": image_drop, "image_attribution_pct": image_attribution},
    "ogm": ogm_summary,
}
with open(os.path.join(MODEL_FOLDER, "robustness_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)

# Explicitly persist named artifacts requested for this experiment.
validation_results_df.to_csv(os.path.join(MODEL_FOLDER, "validation_results.csv"), index=False)
final_summary_df.to_csv(os.path.join(MODEL_FOLDER, "final_summary.csv"), index=False)
print("Artifacts saved to:", MODEL_FOLDER)


# 21. Explainability of the Final Robustness Model



In [ ]:
# Visualization uses overlapping patches for a smooth heatmap.
# faithfulness uses non-overlapping patches as independent perturbation units.

VIS_PATCH_SIZE = 32
VIS_PATCH_STRIDE = 16
FAITH_PATCH_SIZE = 32
FAITH_PATCH_STRIDE = 32

TOKEN_PERTURB_BATCH = 24
IMAGE_PERTURB_BATCH = 16
MAX_EXPLAIN_TOKENS = 96
FAITHFULNESS_FRACTIONS = np.linspace(0.0, 1.0, 11)

EXPLAIN_TARGET_MODE = "predicted"
ROBUST_A4_SUBSTITUTE = TRAIN_SUBSTITUTE
ROBUST_A4_NOISE_SIGMA = 0.0

EXPLAIN_DIR = os.path.join(
    MODEL_FOLDER, "approach4_explainability_robustness_model"
)
os.makedirs(EXPLAIN_DIR, exist_ok=True)

# Confirm that the final robustness model and test outputs are available.
required_names = [
    "model",
    "test_dataset",
    "test_predictions_df",
    "best_threshold",
    "BEST_MODEL_PATH",
    "MODEL_FOLDER",
    "MASKS",
    "constant_mask",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run Sections 1–20 before explainability. "
        f"Missing objects: {missing_names}"
    )

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {BEST_MODEL_PATH}")

if not hasattr(model, "encode") or not hasattr(model, "fuse_and_classify"):
    raise RuntimeError(
        "The loaded model does not provide the robustness-model inference hooks."
    )

if ROBUST_A4_SUBSTITUTE not in ("zero", "calibrated"):
    raise ValueError(f"Unsupported substitution policy: {ROBUST_A4_SUBSTITUTE}")

if not np.isfinite(float(best_threshold)):
    raise RuntimeError("best_threshold is not a valid validation-selected threshold.")

model.eval()
if tokenizer.mask_token_id is None:
    raise ValueError("Word-level occlusion requires a tokenizer mask token.")

# Version counters verify that explainability never mutates model state.
A4_PARAMETER_VERSIONS_BEFORE = {
    name: p._version for name, p in model.named_parameters()
}
A4_BUFFER_VERSIONS_BEFORE = {
    name: b._version for name, b in model.named_buffers()
}

print("Explainability configuration")
print("  checkpoint:", BEST_MODEL_PATH)
print("  target mode:", EXPLAIN_TARGET_MODE)
print("  whole-modality substitute:", ROBUST_A4_SUBSTITUTE)
print("  visualization patch/stride:", VIS_PATCH_SIZE, VIS_PATCH_STRIDE)
print("  faithfulness patch/stride:", FAITH_PATCH_SIZE, FAITH_PATCH_STRIDE)
print("  test samples:", len(test_dataset))
print("  validation-selected threshold:", float(best_threshold))


In [ ]:
# Prediction and confidence utilities

@torch.no_grad()
def _forward_prob_abnormal(
    pixel_values,
    input_ids,
    attention_mask
):
    """
    Return the final robustness model P(abnormal) for a clean multimodal batch.

    No model parameters are changed.
    """

    model.eval()

    with amp_autocast():

        clean_mask = constant_mask(
            pixel_values.shape[0],
            1.0,
            1.0
        )
        logits = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            modality_mask=clean_mask,
            noise_sigma=ROBUST_A4_NOISE_SIGMA,
            substitute=ROBUST_A4_SUBSTITUTE,
        )

    return torch.sigmoid(
        logits.float()
    )

def _target_confidence_from_prob(
    prob_abnormal,
    target_class
):
    """
    Convert P(abnormal) to confidence in the class being explained.

    target_class = 1:
        confidence = P(abnormal)

    target_class = 0:
        confidence = 1 - P(abnormal)
    """

    target_class = int(target_class)

    if target_class == 1:
        return prob_abnormal

    if target_class == 0:
        return 1.0 - prob_abnormal

    raise ValueError(
        "target_class must be 0 or 1."
    )

@torch.no_grad()
def predict_one_from_dataset_index(
    dataset_index
):
    """
    Obtain the robustness model prediction for one test example.
    """

    item = test_dataset[
        int(dataset_index)
    ]

    pixel_values = (
        item["pixel_values"]
        .unsqueeze(0)
        .to(device)
    )

    input_ids = (
        item["input_ids"]
        .unsqueeze(0)
        .to(device)
    )

    attention_mask = (
        item["attention_mask"]
        .unsqueeze(0)
        .to(device)
    )

    prob = float(
        _forward_prob_abnormal(
            pixel_values,
            input_ids,
            attention_mask
        )
        .squeeze()
        .cpu()
    )

    pred = int(
        prob >= best_threshold
    )

    return {
        "dataset_index": int(dataset_index),
        "uid": item["uid"],
        "true_label": int(
            item["label"].item()
        ),
        "prob_abnormal": prob,
        "predicted_label": pred
    }

def choose_target_class(
    predicted_label,
    true_label,
    mode=EXPLAIN_TARGET_MODE
):
    """
    Determine which class should be explained.
    """

    if mode == "predicted":
        return int(predicted_label)

    if mode == "true":
        return int(true_label)

    raise ValueError(
        "mode must be 'predicted' or 'true'."
    )

# WordPiece -> word grouping

def content_word_groups(
    input_ids_1d,
    attention_mask_1d
):
    """
    Group BioClinicalBERT WordPiece tokens into readable words.

    Example:

        ["em", "##phy", "##se", "##ma"]

    becomes:

        {
            "word": "emphysema",
            "positions": [12, 13, 14, 15]
        }

    Punctuation is intentionally preserved because if punctuation
    influences the model, that is itself useful explainability evidence.
    """

    special_ids = set(
        tokenizer.all_special_ids
    )

    ids = (
        input_ids_1d
        .detach()
        .cpu()
        .tolist()
    )

    mask = (
        attention_mask_1d
        .detach()
        .cpu()
        .tolist()
    )

    valid_positions = [
        i
        for i, (tok_id, attended)
        in enumerate(zip(ids, mask))
        if (
            int(attended) == 1
            and int(tok_id)
            not in special_ids
        )
    ]

    valid_positions = (
        valid_positions[
            :MAX_EXPLAIN_TOKENS
        ]
    )

    tokens = [
        tokenizer.convert_ids_to_tokens(
            int(ids[pos])
        )
        for pos in valid_positions
    ]

    groups = []

    for pos, tok in zip(
        valid_positions,
        tokens
    ):

        # WordPiece continuation
        if (
            tok.startswith("##")
            and len(groups) > 0
        ):

            groups[-1]["word"] += (
                tok[2:]
            )

            groups[-1]["positions"].append(
                int(pos)
            )

        else:

            groups.append({
                "word": tok,
                "positions": [
                    int(pos)
                ]
            })

    return groups

# Image patch coordinates

def patch_coordinates(
    height,
    width,
    patch_size,
    stride
):
    """
    Generate fixed-size overlapping image patches.

    Avoids small partial patches at image borders.
    """

    if patch_size > height:
        raise ValueError(
            "patch_size > image height"
        )

    if patch_size > width:
        raise ValueError(
            "patch_size > image width"
        )

    y_starts = list(
        range(
            0,
            height - patch_size + 1,
            stride
        )
    )

    x_starts = list(
        range(
            0,
            width - patch_size + 1,
            stride
        )
    )

    # Guarantee final patch reaches boundary
    if y_starts[-1] != (
        height - patch_size
    ):

        y_starts.append(
            height - patch_size
        )

    if x_starts[-1] != (
        width - patch_size
    ):

        x_starts.append(
            width - patch_size
        )

    coords = []

    for y0 in y_starts:

        for x0 in x_starts:

            coords.append(
                (
                    int(y0),
                    int(y0 + patch_size),
                    int(x0),
                    int(x0 + patch_size)
                )
            )

    return coords

# AUC helper

def auc_over_fraction(
    fractions,
    confidences
):
    """
    Trapezoidal area under confidence-vs-fraction curve.
    """

    x = np.asarray(
        fractions,
        dtype=float
    )

    y = np.asarray(
        confidences,
        dtype=float
    )

    return float(
        np.trapezoid(
            y,
            x
        )
    )

print(
    "Robust-model explainability utilities ready."
)


In [ ]:
# Prediction integrity check

required_prediction_cols = {
    "uid", "true_label", "prob_multimodal", "pred_multimodal"
}
missing_prediction_cols = required_prediction_cols.difference(test_predictions_df.columns)
if missing_prediction_cols:
    raise RuntimeError(
        "test_predictions_df does not contain the robustness-model prediction columns: "
        f"{sorted(missing_prediction_cols)}"
    )

check_dataset_indices = sorted(set([
    0,
    len(test_dataset) // 2,
    len(test_dataset) - 1,
]))

probability_differences = []
for dataset_index in check_dataset_indices:
    rec = predict_one_from_dataset_index(dataset_index)
    uid_key = str(rec["uid"])
    stored = test_predictions_df.loc[
        test_predictions_df["uid"].astype(str) == uid_key,
        "prob_multimodal",
    ]
    if len(stored) != 1:
        raise RuntimeError(
            f"Expected exactly one stored robustness prediction for UID {rec['uid']}; "
            f"found {len(stored)}."
        )
    stored_prob = float(stored.iloc[0])
    diff = abs(float(rec["prob_abnormal"]) - stored_prob)
    probability_differences.append(diff)
    print(
        f"UID={rec['uid']} | direct={rec['prob_abnormal']:.6f} | "
        f"stored robust={stored_prob:.6f} | abs diff={diff:.2e}"
    )

max_prediction_diff = max(probability_differences) if probability_differences else 0.0
if max_prediction_diff > 1e-3:
    raise RuntimeError(
        "Approach-4 clean inference does not match the robustness-model test predictions. "
        f"Maximum absolute probability difference: {max_prediction_diff:.6g}. "
        "Check checkpoint loading, preprocessing, threshold, and model state before continuing."
    )

print(
    "Integrity check passed: Approach 4 is explaining the same clean robustness-model "
    "predictions used in the reported test results."
)


In [ ]:
# Word-level text attribution

@torch.no_grad()
def explain_text_words(
    pixel_values,
    input_ids,
    attention_mask,
    target_class,
    perturb_batch_size=TOKEN_PERTURB_BATCH
):
    """
    Word-level text occlusion.

    All WordPiece tokens belonging to one word are masked together.

    Importance(word):

        confidence(original)
        -
        confidence(word masked)

    Positive importance:
        word supports explained prediction.

    Negative importance:
        masking the word increases confidence in explained prediction.
    """

    model.eval()

    if pixel_values.shape[0] != 1:

        raise ValueError(
            "explain_text_words expects batch size 1."
        )

    # Original prediction confidence

    base_prob = float(
        _forward_prob_abnormal(
            pixel_values,
            input_ids,
            attention_mask
        )
        .squeeze()
        .cpu()
    )

    base_conf = float(
        _target_confidence_from_prob(
            base_prob,
            target_class
        )
    )

    # Obtain grouped words

    groups = content_word_groups(
        input_ids[0],
        attention_mask[0]
    )

    if len(groups) == 0:
        return pd.DataFrame(
            columns=[
                "word",
                "positions",
                "base_confidence",
                "occluded_confidence",
                "importance"
            ]
        )

    rows = []

    # Perturb grouped words in batches

    for start in range(
        0,
        len(groups),
        perturb_batch_size
    ):

        chunk = groups[
            start:
            start + perturb_batch_size
        ]

        b = len(chunk)

        ids_batch = input_ids.repeat(b,1)
        mask_batch = attention_mask.repeat(b,1)
        image_batch = pixel_values.repeat(b,1,1,1)

        # Mask all WordPieces belonging to each word
        for j, group in enumerate(chunk):

            ids_batch[
                j,
                group["positions"]
            ] = tokenizer.mask_token_id

        probs = (
            _forward_prob_abnormal(
                image_batch,
                ids_batch,
                mask_batch
            )
            .detach()
            .cpu()
            .numpy()
            .ravel()
        )

        for group, prob in zip(
            chunk,
            probs
        ):

            occ_conf = float(
                _target_confidence_from_prob(
                    float(prob),
                    target_class
                )
            )

            rows.append({
                "word": group["word"],
                "positions": group["positions"],
                "base_confidence": base_conf,
                "occluded_confidence": occ_conf,
                "importance": (
                    base_conf
                    -
                    occ_conf
                )
            })

    word_df = pd.DataFrame(
        rows
    )

    return (
        word_df
        .sort_values(
            "importance",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

print(
    "Robust-model word-level text occlusion explainer ready."
)


In [ ]:
# Image-patch attribution

@torch.no_grad()
def explain_image_patches(
    pixel_values,
    input_ids,
    attention_mask,
    target_class,
    patch_size=VIS_PATCH_SIZE,
    stride=VIS_PATCH_STRIDE,
    perturb_batch_size=IMAGE_PERTURB_BATCH
):
    """
    One-patch-at-a-time image occlusion.

    Image patches overlap because stride < patch_size.

    Importance(patch):

        confidence(original)
        -
        confidence(patch removed)

    Positive:
        patch supports explained prediction.

    Negative:
        removing patch increases confidence.

    Returns
    -------
    patch_df:
        one row per image patch

    heatmap:
        pixel-level average occlusion importance map
    """

    model.eval()

    if pixel_values.shape[0] != 1:

        raise ValueError(
            "explain_image_patches expects batch size 1."
        )

    _, _, h, w = (
        pixel_values.shape
    )

    base_prob = float(
        _forward_prob_abnormal(
            pixel_values,
            input_ids,
            attention_mask
        )
        .squeeze()
        .cpu()
    )

    base_conf = float(
        _target_confidence_from_prob(
            base_prob,
            target_class
        )
    )

    coords = patch_coordinates(
        h,
        w,
        patch_size,
        stride
    )

    rows = []

    # Used to average overlapping patches
    heat_sum = np.zeros(
        (h, w),
        dtype=np.float32
    )

    heat_count = np.zeros(
        (h, w),
        dtype=np.float32
    )

    for start in range(
        0,
        len(coords),
        perturb_batch_size
    ):

        chunk = coords[
            start:
            start + perturb_batch_size
        ]

        b = len(chunk)

        image_batch = pixel_values.repeat(
            b,
            1,
            1,
            1
        )

        ids_batch = input_ids.repeat(
            b,
            1
        )

        mask_batch = attention_mask.repeat(
            b,
            1
        )

        # Zero one normalized image patch
        for j, (
            y0,
            y1,
            x0,
            x1
        ) in enumerate(chunk):

            image_batch[
                j,
                :,
                y0:y1,
                x0:x1
            ] = 0.0

        probs = (
            _forward_prob_abnormal(
                image_batch,
                ids_batch,
                mask_batch
            )
            .detach()
            .cpu()
            .numpy()
            .ravel()
        )

        for (
            y0,
            y1,
            x0,
            x1
        ), prob in zip(
            chunk,
            probs
        ):

            occ_conf = float(
                _target_confidence_from_prob(
                    float(prob),
                    target_class
                )
            )

            score = (
                base_conf
                -
                occ_conf
            )

            rows.append({
                "y0": int(y0),
                "y1": int(y1),
                "x0": int(x0),
                "x1": int(x1),
                "base_confidence": base_conf,
                "occluded_confidence": occ_conf,
                "importance": score
            })

            heat_sum[
                y0:y1,
                x0:x1
            ] += score

            heat_count[
                y0:y1,
                x0:x1
            ] += 1.0

    heatmap = (
        heat_sum
        /
        np.maximum(
            heat_count,
            1.0
        )
    )

    patch_df = pd.DataFrame(
        rows
    )

    patch_df = (
        patch_df
        .sort_values(
            "importance",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    return (
        patch_df,
        heatmap
    )

print(
    "Robust-model overlapping image-patch explainer ready."
)


In [ ]:
# Whole-modality importance

@torch.no_grad()
def modality_level_importance(
    pixel_values,
    input_ids,
    attention_mask,
    target_class,
    substitute=ROBUST_A4_SUBSTITUTE,
):
    """
    Whole-modality confidence effects for the robustness-trained model.

    This uses the model's native `encode` + `fuse_and_classify` pathway.
    Missing modalities are applied to the 768-D encoder-level modality embeddings
    before CBP using the same zero-substitution policy used by the primary robustness
    training/evaluation experiment. No latent noise is added here.

    Image importance = confidence(full) - confidence(text-only)
    Text importance  = confidence(full) - confidence(image-only)
    """
    model.eval()

    with amp_autocast():
        text_features, image_features = model.encode(
            pixel_values, input_ids, attention_mask
        )

    probs = {}
    for condition, (text_present, image_present) in MASKS.items():
        mask = constant_mask(
            text_features.shape[0], text_present, image_present
        )
        with amp_autocast():
            logits = model.fuse_and_classify(
                text_features,
                image_features,
                modality_mask=mask,
                noise_sigma=ROBUST_A4_NOISE_SIGMA,
                substitute=substitute,
            )
        probs[condition] = float(torch.sigmoid(logits.float()).squeeze().cpu())

    def conf(p):
        return float(_target_confidence_from_prob(p, target_class))

    full_conf = conf(probs["multimodal"])
    text_only_conf = conf(probs["text_only"])
    image_only_conf = conf(probs["image_only"])

    return {
        "full_target_confidence": full_conf,
        "image_removed_target_confidence": text_only_conf,
        "text_removed_target_confidence": image_only_conf,
        "image_modality_importance": full_conf - text_only_conf,
        "text_modality_importance": full_conf - image_only_conf,
        "substitute": substitute,
    }

print("Robust-model native whole-modality explainer ready.")


In [ ]:
# Deletion and insertion faithfulness

@torch.no_grad()
def _confidence_for_single_inputs(
    pixel_values,
    input_ids,
    attention_mask,
    target_class
):
    """
    Confidence in the class being explained.
    """

    p = float(
        _forward_prob_abnormal(
            pixel_values,
            input_ids,
            attention_mask
        )
        .squeeze()
        .cpu()
    )

    return float(
        _target_confidence_from_prob(
            p,
            target_class
        )
    )

@torch.no_grad()
def text_faithfulness_curves(
    pixel_values,
    input_ids,
    attention_mask,
    target_class,
    word_df,
    fractions=FAITHFULNESS_FRACTIONS
):
    """
    Text deletion/insertion while original image remains available.

    Deletion:
        progressively mask highest-ranked words.

    Insertion:
        start with all content words masked and progressively restore
        highest-ranked words.
    """

    positive_df = word_df[
        word_df["importance"] > 0
    ]

    if len(positive_df) > 0:

        ranked_groups = (
            positive_df[
                "positions"
            ]
            .tolist()
        )

    else:

        ranked_groups = (
            word_df[
                "positions"
            ]
            .tolist()
        )

    # All content positions
    all_groups = content_word_groups(
        input_ids[0],
        attention_mask[0]
    )

    all_content_positions = [
        pos
        for group in all_groups
        for pos in group["positions"]
    ]

    k_total = len(
        ranked_groups
    )

    deletion_conf = []
    insertion_conf = []

    for frac in fractions:

        k = int(
            round(
                float(frac)
                *
                k_total
            )
        )

        chosen_groups = (
            ranked_groups[:k]
        )

        chosen_positions = [
            pos
            for group_positions
            in chosen_groups
            for pos
            in group_positions
        ]

        # DELETION

        del_ids = (
            input_ids.clone()
        )

        if chosen_positions:

            del_ids[
                0,
                chosen_positions
            ] = tokenizer.mask_token_id

        deletion_conf.append(
            _confidence_for_single_inputs(
                pixel_values,
                del_ids,
                attention_mask,
                target_class
            )
        )

        # INSERTION

        ins_ids = (
            input_ids.clone()
        )

        # Mask all content words first
        if all_content_positions:

            ins_ids[
                0,
                all_content_positions
            ] = tokenizer.mask_token_id

        # Restore selected words
        if chosen_positions:

            ins_ids[
                0,
                chosen_positions
            ] = input_ids[
                0,
                chosen_positions
            ]

        insertion_conf.append(
            _confidence_for_single_inputs(
                pixel_values,
                ins_ids,
                attention_mask,
                target_class
            )
        )

    return {

        "fractions":
            np.asarray(
                fractions,
                dtype=float
            ),

        "deletion_confidence":
            np.asarray(
                deletion_conf,
                dtype=float
            ),

        "insertion_confidence":
            np.asarray(
                insertion_conf,
                dtype=float
            ),

        "deletion_auc":
            auc_over_fraction(
                fractions,
                deletion_conf
            ),

        "insertion_auc":
            auc_over_fraction(
                fractions,
                insertion_conf
            )
    }

@torch.no_grad()
def image_faithfulness_curves(
    pixel_values,
    input_ids,
    attention_mask,
    target_class,
    patch_df,
    fractions=FAITHFULNESS_FRACTIONS
):
    """
    Image deletion/insertion while original text remains available.
    """

    positive_df = patch_df[
        patch_df["importance"] > 0
    ]

    if len(positive_df) > 0:
        ranked = positive_df.copy()
    else:
        ranked = patch_df.copy()

    coords = list(
        ranked[
            [
                "y0",
                "y1",
                "x0",
                "x1"
            ]
        ]
        .astype(int)
        .itertuples(
            index=False,
            name=None
        )
    )

    k_total = len(coords)

    deletion_conf = []
    insertion_conf = []

    for frac in fractions:

        k = int(
            round(
                float(frac)
                *
                k_total
            )
        )

        chosen = coords[:k]

        # DELETION

        del_img = (
            pixel_values.clone()
        )

        for (
            y0,
            y1,
            x0,
            x1
        ) in chosen:

            del_img[
                :,
                :,
                y0:y1,
                x0:x1
            ] = 0.0

        deletion_conf.append(
            _confidence_for_single_inputs(
                del_img,
                input_ids,
                attention_mask,
                target_class
            )
        )

        # INSERTION

        ins_img = torch.zeros_like(
            pixel_values
        )

        for (
            y0,
            y1,
            x0,
            x1
        ) in chosen:

            ins_img[
                :,
                :,
                y0:y1,
                x0:x1
            ] = pixel_values[
                :,
                :,
                y0:y1,
                x0:x1
            ]

        insertion_conf.append(
            _confidence_for_single_inputs(
                ins_img,
                input_ids,
                attention_mask,
                target_class
            )
        )

    return {

        "fractions":
            np.asarray(
                fractions,
                dtype=float
            ),

        "deletion_confidence":
            np.asarray(
                deletion_conf,
                dtype=float
            ),

        "insertion_confidence":
            np.asarray(
                insertion_conf,
                dtype=float
            ),

        "deletion_auc":
            auc_over_fraction(
                fractions,
                deletion_conf
            ),

        "insertion_auc":
            auc_over_fraction(
                fractions,
                insertion_conf
            )
    }

@torch.no_grad()
def multimodal_faithfulness_curves(
    pixel_values,
    input_ids,
    attention_mask,
    target_class,
    word_df,
    patch_df,
    fractions=FAITHFULNESS_FRACTIONS
):
    """
    Joint text + image deletion/insertion.

    Words and image patches are ranked together using their
    one-feature confidence-drop importance.

    Important:
        this is a perturbation ranking, not a Shapley decomposition.
    """

    units = []

    # Add word units

    for row in word_df.itertuples(
        index=False
    ):

        units.append({

            "kind":
                "word",

            "importance":
                float(
                    row.importance
                ),

            "positions":
                list(
                    row.positions
                )
        })

    # Add image patch units

    for row in patch_df.itertuples(
        index=False
    ):

        units.append({

            "kind":
                "patch",

            "importance":
                float(
                    row.importance
                ),

            "coord":
                (
                    int(row.y0),
                    int(row.y1),
                    int(row.x0),
                    int(row.x1)
                )
        })

    positive_units = [
        unit
        for unit in units
        if unit["importance"] > 0
    ]

    if len(positive_units) > 0:

        ranked_units = sorted(
            positive_units,
            key=lambda x:
                x["importance"],
            reverse=True
        )

    else:

        ranked_units = sorted(
            units,
            key=lambda x:
                x["importance"],
            reverse=True
        )

    # Every original text position

    all_groups = content_word_groups(
        input_ids[0],
        attention_mask[0]
    )

    all_content_positions = [
        pos
        for group in all_groups
        for pos
        in group["positions"]
    ]

    k_total = len(
        ranked_units
    )

    deletion_conf = []
    insertion_conf = []

    for frac in fractions:

        k = int(
            round(
                float(frac)
                *
                k_total
            )
        )

        chosen = (
            ranked_units[:k]
        )

        # JOINT DELETION

        del_img = (
            pixel_values.clone()
        )

        del_ids = (
            input_ids.clone()
        )

        for unit in chosen:

            if (
                unit["kind"]
                ==
                "word"
            ):

                del_ids[
                    0,
                    unit["positions"]
                ] = tokenizer.mask_token_id

            else:

                (
                    y0,
                    y1,
                    x0,
                    x1
                ) = unit["coord"]

                del_img[
                    :,
                    :,
                    y0:y1,
                    x0:x1
                ] = 0.0

        deletion_conf.append(
            _confidence_for_single_inputs(
                del_img,
                del_ids,
                attention_mask,
                target_class
            )
        )

        # JOINT INSERTION

        ins_img = torch.zeros_like(
            pixel_values
        )

        ins_ids = (
            input_ids.clone()
        )

        # Start from completely masked content text
        if all_content_positions:

            ins_ids[
                0,
                all_content_positions
            ] = tokenizer.mask_token_id

        for unit in chosen:

            if (
                unit["kind"]
                ==
                "word"
            ):

                positions = (
                    unit["positions"]
                )

                ins_ids[
                    0,
                    positions
                ] = input_ids[
                    0,
                    positions
                ]

            else:

                (
                    y0,
                    y1,
                    x0,
                    x1
                ) = unit["coord"]

                ins_img[
                    :,
                    :,
                    y0:y1,
                    x0:x1
                ] = pixel_values[
                    :,
                    :,
                    y0:y1,
                    x0:x1
                ]

        insertion_conf.append(
            _confidence_for_single_inputs(
                ins_img,
                ins_ids,
                attention_mask,
                target_class
            )
        )

    return {

        "fractions":
            np.asarray(
                fractions,
                dtype=float
            ),

        "deletion_confidence":
            np.asarray(
                deletion_conf,
                dtype=float
            ),

        "insertion_confidence":
            np.asarray(
                insertion_conf,
                dtype=float
            ),

        "deletion_auc":
            auc_over_fraction(
                fractions,
                deletion_conf
            ),

        "insertion_auc":
            auc_over_fraction(
                fractions,
                insertion_conf
            )
    }

print(
    "Robust-model word/image/joint faithfulness evaluators ready."
)


In [ ]:
# Visualization utilities

def denormalize_image(
    pixel_values_1
):
    """Convert one DenseNet-normalized (1,C,H,W) tensor to displayable RGB."""
    x = pixel_values_1[0].detach().float().cpu()
    mean_t = torch.tensor(DENSENET_IMAGE_MEAN, dtype=x.dtype).view(-1, 1, 1)
    std_t = torch.tensor(DENSENET_IMAGE_STD, dtype=x.dtype).view(-1, 1, 1)
    x = x * std_t + mean_t
    x = x.clamp(0.0, 1.0)
    return x.permute(1, 2, 0).numpy()

def plot_explanation(
    record,
    image_heatmap,
    word_df,
    image_curves=None,
    text_curves=None,
    joint_curves=None,
    save_path=None,
    top_words=12
):
    """
    Plot:

    1. Original X-ray
    2. Signed image occlusion heatmap
    3. Most influential words
    4. Faithfulness curves
    """

    rgb = (
        record[
            "display_image"
        ]
    )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15, 11)
    )

    axes[0, 0].imshow(
        rgb,
        cmap="gray"
    )

    axes[0, 0].set_title(

        f"UID {record['uid']} | "
        f"true={record['true_label']} | "
        f"pred={record['predicted_label']} | "
        f"P(abn)={record['prob_abnormal']:.3f}"
    )

    axes[0, 0].axis(
        "off"
    )

    axes[0, 1].imshow(
        rgb,
        cmap="gray"
    )

    max_abs = float(
        np.max(
            np.abs(
                image_heatmap
            )
        )
    )

    if max_abs > 0:

        hm = (
            image_heatmap
            /
            (
                max_abs
                +
                1e-12
            )
        )

        overlay = axes[
            0,
            1
        ].imshow(
            hm,
            cmap="coolwarm",
            alpha=0.50,
            vmin=-1,
            vmax=1
        )

        fig.colorbar(
            overlay,
            ax=axes[0, 1],
            fraction=0.046,
            pad=0.04,
            label="Signed occlusion importance"
        )

    axes[0, 1].set_title(

        f"Signed image evidence for class "
        f"{record['target_class']}\n"
        "Red = supports | Blue = opposes"
    )

    axes[0, 1].axis(
        "off"
    )

    wd = (
        word_df
        .head(
            top_words
        )
        .copy()
    )

    if len(wd) > 0:

        # Reverse so largest bar appears at top
        wd = wd.iloc[
            ::-1
        ]

        axes[1, 0].barh(
            wd["word"],
            wd["importance"]
        )

        axes[1, 0].axvline(
            0.0,
            linewidth=1
        )

        axes[1, 0].set_xlabel(
            "Target-confidence drop when word is masked"
        )

        axes[1, 0].set_title(
            "Most influential report words"
        )

    else:

        axes[1, 0].text(
            0.5,
            0.5,
            "No explainable content words",
            ha="center",
            va="center"
        )

        axes[1, 0].set_axis_off()

    ax = axes[
        1,
        1
    ]

    if joint_curves is not None:

        ax.plot(
            joint_curves[
                "fractions"
            ],
            joint_curves[
                "deletion_confidence"
            ],
            marker="o",
            label=(
                "Joint deletion "
                f"(AUC="
                f"{joint_curves['deletion_auc']:.3f})"
            )
        )

        ax.plot(
            joint_curves[
                "fractions"
            ],
            joint_curves[
                "insertion_confidence"
            ],
            marker="o",
            label=(
                "Joint insertion "
                f"(AUC="
                f"{joint_curves['insertion_auc']:.3f})"
            )
        )

    if image_curves is not None:

        ax.plot(
            image_curves[
                "fractions"
            ],
            image_curves[
                "deletion_confidence"
            ],
            linestyle="--",
            label=(
                "Image deletion "
                f"(AUC="
                f"{image_curves['deletion_auc']:.3f})"
            )
        )

    if text_curves is not None:

        ax.plot(
            text_curves[
                "fractions"
            ],
            text_curves[
                "deletion_confidence"
            ],
            linestyle=":",
            label=(
                "Text deletion "
                f"(AUC="
                f"{text_curves['deletion_auc']:.3f})"
            )
        )

    ax.set_xlabel(
        "Fraction of ranked features perturbed"
    )

    ax.set_ylabel(
        "Confidence in explained class"
    )

    ax.set_ylim(
        0,
        1
    )

    ax.set_title(
        "Deletion / insertion faithfulness"
    )

    ax.legend(
        fontsize=8
    )

    plt.tight_layout()

    if save_path is not None:

        fig.savefig(
            save_path,
            dpi=180,
            bbox_inches="tight"
        )

    plt.show()

    plt.close(
        fig
    )

print(
    "Robust-model visualization utilities ready."
)


In [ ]:
# Prepare the complete held-out test set for explainability

required_cols = {"uid", "true_label", "prob_multimodal", "pred_multimodal"}
missing_cols = required_cols.difference(test_predictions_df.columns)
if missing_cols:
    raise RuntimeError(
        f"Missing required prediction columns: {sorted(missing_cols)}. "
        "Run Sections 1–20 before explainability."
    )

uid_to_dataset_index = {
    str(test_dataset[i]["uid"]): i for i in range(len(test_dataset))
}

selection_df = test_predictions_df.copy()
selection_df["dataset_index"] = (
    selection_df["uid"].astype(str).map(uid_to_dataset_index)
)

if selection_df["dataset_index"].isna().any():
    missing = selection_df.loc[
        selection_df["dataset_index"].isna(), "uid"
    ].tolist()[:10]
    raise RuntimeError(f"Could not map test UIDs to test_dataset. Examples: {missing}")

selection_df["dataset_index"] = selection_df["dataset_index"].astype(int)
selection_df["prob_abnormal"] = selection_df["prob_multimodal"].astype(float)
selection_df["predicted_label"] = selection_df["pred_multimodal"].astype(int)

selection_df["error_group"] = np.select(
    [
        (selection_df["true_label"] == 1) & (selection_df["predicted_label"] == 1),
        (selection_df["true_label"] == 0) & (selection_df["predicted_label"] == 0),
        (selection_df["true_label"] == 0) & (selection_df["predicted_label"] == 1),
        (selection_df["true_label"] == 1) & (selection_df["predicted_label"] == 0),
    ],
    ["TP", "TN", "FP", "FN"],
    default="OTHER",
)

# Final explainability analysis uses the complete held-out test set.
selected_explain_df = selection_df.sort_values("dataset_index").reset_index(drop=True)

print(f"Explainability samples: {len(selected_explain_df)}")
display(
    selected_explain_df["error_group"]
    .value_counts()
    .reindex(["TP", "TN", "FP", "FN"], fill_value=0)
    .rename("count")
    .to_frame()
)


In [ ]:
# End-to-end sample explainer

@torch.no_grad()
def explain_dataset_index(dataset_index, compute_curves=True):
    """Return text, image, modality, and faithfulness explanations for one test case."""
    item = test_dataset[int(dataset_index)]

    pixel_values = item["pixel_values"].unsqueeze(0).to(device)
    input_ids = item["input_ids"].unsqueeze(0).to(device)
    attention_mask = item["attention_mask"].unsqueeze(0).to(device)

    prob_abnormal = float(
        _forward_prob_abnormal(pixel_values, input_ids, attention_mask)
        .squeeze()
        .cpu()
    )
    predicted_label = int(prob_abnormal >= best_threshold)
    true_label = int(item["label"].item())
    target_class = choose_target_class(
        predicted_label=predicted_label,
        true_label=true_label,
        mode=EXPLAIN_TARGET_MODE,
    )
    target_confidence = float(
        _target_confidence_from_prob(prob_abnormal, target_class)
    )

    # Text attribution.
    word_df = explain_text_words(
        pixel_values, input_ids, attention_mask, target_class
    )

    # Overlapping patches are used only for the smooth visualization heatmap.
    patch_df_vis, image_heatmap = explain_image_patches(
        pixel_values,
        input_ids,
        attention_mask,
        target_class,
        patch_size=VIS_PATCH_SIZE,
        stride=VIS_PATCH_STRIDE,
    )

    # Non-overlapping patches are used for quantitative faithfulness.
    patch_df_faith, _ = explain_image_patches(
        pixel_values,
        input_ids,
        attention_mask,
        target_class,
        patch_size=FAITH_PATCH_SIZE,
        stride=FAITH_PATCH_STRIDE,
    )

    modality_importance = modality_level_importance(
        pixel_values, input_ids, attention_mask, target_class
    )

    text_curves = image_curves = joint_curves = None
    if compute_curves:
        text_curves = text_faithfulness_curves(
            pixel_values, input_ids, attention_mask, target_class, word_df
        )
        image_curves = image_faithfulness_curves(
            pixel_values, input_ids, attention_mask, target_class, patch_df_faith
        )
        joint_curves = multimodal_faithfulness_curves(
            pixel_values,
            input_ids,
            attention_mask,
            target_class,
            word_df,
            patch_df_faith,
        )

    record = {
        "dataset_index": int(dataset_index),
        "uid": item["uid"],
        "true_label": true_label,
        "predicted_label": predicted_label,
        "target_class": target_class,
        "prob_abnormal": prob_abnormal,
        "target_confidence": target_confidence,
        "display_image": denormalize_image(pixel_values),
    }

    return {
        "record": record,
        "word_df": word_df,
        "patch_df": patch_df_vis,
        "patch_df_faith": patch_df_faith,
        "image_heatmap": image_heatmap,
        "modality_importance": modality_importance,
        "text_curves": text_curves,
        "image_curves": image_curves,
        "joint_curves": joint_curves,
    }

print("End-to-end explainer ready.")


In [ ]:
# Single-UID inspection helper

def inspect_robust_uid(target_uid, top_words=12, top_patches=10):
    """Display a complete explanation for one test UID."""
    match = selection_df[selection_df["uid"].astype(str) == str(target_uid)]
    if match.empty:
        raise ValueError(f"UID {target_uid} was not found in the test set.")

    dataset_index = int(match.iloc[0]["dataset_index"])
    error_group = str(match.iloc[0]["error_group"])
    ex = explain_dataset_index(dataset_index, compute_curves=True)
    rec = ex["record"]
    tc, ic, jc = ex["text_curves"], ex["image_curves"], ex["joint_curves"]

    print(
        f"UID={rec['uid']} | {error_group} | true={rec['true_label']} | "
        f"pred={rec['predicted_label']} | P(abnormal)={rec['prob_abnormal']:.4f} | "
        f"explained class={rec['target_class']}"
    )

    print("\nWhole-modality importance:")
    display(pd.DataFrame([ex["modality_importance"]]))

    print("\nTop influential report words:")
    display(
        ex["word_df"][
            ["word", "importance", "base_confidence", "occluded_confidence"]
        ].head(top_words)
    )

    print("\nTop quantitative image patches:")
    display(
        ex["patch_df_faith"][
            ["y0", "y1", "x0", "x1", "importance", "base_confidence", "occluded_confidence"]
        ].head(top_patches)
    )

    print(
        "\nFaithfulness AUCs "
        f"| text: {tc['deletion_auc']:.4f}/{tc['insertion_auc']:.4f} "
        f"| image: {ic['deletion_auc']:.4f}/{ic['insertion_auc']:.4f} "
        f"| joint: {jc['deletion_auc']:.4f}/{jc['insertion_auc']:.4f}"
    )

    plot_explanation(
        rec,
        ex["image_heatmap"],
        ex["word_df"],
        image_curves=ic,
        text_curves=tc,
        joint_curves=jc,
        save_path=None,
        top_words=min(top_words, 15),
    )
    return ex


In [ ]:
# Full-test quantitative explainability evaluation

faithfulness_rows, all_word_rows, all_patch_rows = [], [], []

for row in tqdm(
    selected_explain_df.itertuples(index=False),
    total=len(selected_explain_df),
    desc="Explainability evaluation",
):
    ex = explain_dataset_index(int(row.dataset_index), compute_curves=True)
    rec = ex["record"]
    tc, ic, jc = ex["text_curves"], ex["image_curves"], ex["joint_curves"]
    mi = ex["modality_importance"]

    idx_20 = int(np.argmin(np.abs(jc["fractions"] - 0.20)))
    top20_drop = float(
        jc["deletion_confidence"][0] - jc["deletion_confidence"][idx_20]
    )

    faithfulness_rows.append(
        {
            "uid": rec["uid"],
            "dataset_index": rec["dataset_index"],
            "error_group": row.error_group,
            "true_label": rec["true_label"],
            "predicted_label": rec["predicted_label"],
            "target_class": rec["target_class"],
            "prob_abnormal": rec["prob_abnormal"],
            "target_confidence": rec["target_confidence"],
            "text_deletion_auc": tc["deletion_auc"],
            "text_insertion_auc": tc["insertion_auc"],
            "image_deletion_auc": ic["deletion_auc"],
            "image_insertion_auc": ic["insertion_auc"],
            "joint_deletion_auc": jc["deletion_auc"],
            "joint_insertion_auc": jc["insertion_auc"],
            "joint_top20_confidence_drop": top20_drop,
            "image_modality_importance": mi["image_modality_importance"],
            "text_modality_importance": mi["text_modality_importance"],
            "n_text_words": len(ex["word_df"]),
            "n_image_patches": len(ex["patch_df_faith"]),
        }
    )

    word_rows = ex["word_df"].copy()
    word_rows.insert(0, "uid", rec["uid"])
    word_rows.insert(1, "error_group", row.error_group)
    all_word_rows.append(word_rows)

    patch_rows = ex["patch_df_faith"].copy()
    patch_rows.insert(0, "uid", rec["uid"])
    patch_rows.insert(1, "error_group", row.error_group)
    all_patch_rows.append(patch_rows)

faithfulness_df = pd.DataFrame(faithfulness_rows)
word_attributions_df = (
    pd.concat(all_word_rows, ignore_index=True) if all_word_rows else pd.DataFrame()
)
patch_attributions_df = (
    pd.concat(all_patch_rows, ignore_index=True) if all_patch_rows else pd.DataFrame()
)

metric_columns = [
    "text_deletion_auc",
    "text_insertion_auc",
    "image_deletion_auc",
    "image_insertion_auc",
    "joint_deletion_auc",
    "joint_insertion_auc",
    "joint_top20_confidence_drop",
    "image_modality_importance",
    "text_modality_importance",
]

faithfulness_summary_df = pd.DataFrame(
    [
        {
            "Metric": col,
            "Mean": faithfulness_df[col].mean(),
            "SD": faithfulness_df[col].std(ddof=1),
            "Median": faithfulness_df[col].median(),
            "N": faithfulness_df[col].notna().sum(),
        }
        for col in metric_columns
    ]
)

group_summary_mean = (
    faithfulness_df.groupby("error_group")[metric_columns].mean().round(4)
)
group_summary_sd = (
    faithfulness_df.groupby("error_group")[metric_columns].std().round(4)
)

print("Overall explainability summary:")
display(faithfulness_summary_df.round(6))

print("\nMean metrics by prediction group:")
display(group_summary_mean)

print(
    "\nInterpretation: lower deletion AUC is better; higher insertion AUC is better. "
    "Positive modality importance means that removing the modality reduced confidence "
    "in the explained class."
)
